In [1]:
from dotenv import load_dotenv
import os

load_dotenv("backend/.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.0-flash")

print("API key loaded:", bool(GOOGLE_API_KEY))

API key loaded: True


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

In [3]:
def check_grounding(question, context, answer):
    
    prompt = f"""
You are a grounding checker.

Your job is to check whether EVERY factual claim in the answer
is supported by the provided context.

Rules:
- Use ONLY the provided context.
- Do not use your own knowledge.
- Identify every unsupported factual claim.
- If every claim is supported, return exactly:

SUPPORTED

- If one or more claims are unsupported, return:

UNSUPPORTED
Claim: <unsupported claim>
Reason: <why it is not supported by the context>

Question:
{question}

Context:
{context}

Answer:
{answer}
"""

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, str):
        return content.strip()

    if isinstance(content, list):
        parts = []

        for block in content:
            if isinstance(block, str):
                parts.append(block)

            elif isinstance(block, dict):
                if "text" in block:
                    parts.append(block["text"])

        return "".join(parts).strip()

    return str(content).strip()

In [4]:
question = "What is the ARR growth rate?"

context = """
According to the internal document, the ARR growth rate is 25%.
"""

answer = """
The ARR growth rate is 25%, and the company expects revenue
to increase by 40%.
"""

result = check_grounding(question, context, answer)

print(result)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


UNSUPPORTED
Claim: The company expects revenue to increase by 40%.
Reason: The provided context only mentions the ARR growth rate and does not contain any information regarding expected revenue growth.
